# Removing highly correlated variables

This small intermediate chapter straddles the limit between feature selection and feature engineering.

Highly correlated variables can cause problem in model fitting and feature selection because they contain  redundant information. 

Here we will apply a relatively simple procedure: detect clusters of highly correlated variables, and keep only 1 variable per cluster.

> NB: this is inspired by an [sklearn example](https://scikit-learn.org/stable/auto_examples/inspection/plot_permutation_importance_multicollinear.html) and [feature-engine's DropCorrelatedFeatures function](https://feature-engine.trainindata.com/en/latest/api_doc/selection/DropCorrelatedFeatures.html#feature_engine.selection.DropCorrelatedFeatures)

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectPercentile
import numpy as np

## loading data
df_xpr = pd.read_csv("../data/TGCA_BRCA_expression_matrix.TPM.csv.gz" , index_col = 0)

df_clinical = pd.read_csv("../data/TGCA_BRCA_clinical_filtered.small.csv",index_col=0)
df_clinical = pd.get_dummies( df_clinical , drop_first=True)

## y is the poor_diagnosis
y = df_clinical.poor_prognosis

## ensuring the expression data is properly ordered
X_xpr = df_xpr.loc[ :, df_clinical.index].transpose() 

## selecting top 1% most variable genes
VT = SelectPercentile( score_func = lambda x,_ : np.var(x , axis = 0) ,
                       percentile = 0.1
                     )

X = pd.DataFrame( VT.fit_transform(X_xpr), columns=VT.get_feature_names_out() , index = X_xpr.index )

X.shape

Let's look at the linear correlation among these genes:

In [ ]:
import seaborn as sns

sns.clustermap(X.corr(), square=True, cmap="RdBu_r", vmin=-1, vmax=1,
            xticklabels=False, yticklabels=False)

In [ ]:
import matplotlib.pyplot as plt
C = np.array( X.corr() )
sns.histplot( C[ *( np.tril_indices_from( C ,  k = -1 ) ) ] )
plt.xlabel("feature correlation")

We can clearly see some pattern emerge, but more importantly some variables with very high correlations.

Let's flag them visually:

In [ ]:
threshold = 0.9
sns.clustermap(X.corr()>threshold, square=True, cmap="RdBu_r", vmin=-1, vmax=1,
            xticklabels=False, yticklabels=False)

Grouping the variables comes down to hierarchical clustering using absolute correlation as a metric with single linkage.

> NB: correlation is not a distance, so we actually use 1 - abs(correlation)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

corr_threshold = 0.9

metric = 1 - X.corr().abs()

HC = AgglomerativeClustering( n_clusters=None , metric='precomputed', linkage = 'single' , distance_threshold = (1-corr_threshold) )
HC.fit(metric)

variable_clusters = pd.Series( HC.labels_  , index = X.columns)


print(f"{len(variable_clusters)} in {variable_clusters.nunique()} clusters")

variable_clusters.value_counts().iloc[:7]

From there we would keep a single feature per cluster

In [ ]:
variable_clusters.index.groupby(variable_clusters)

In [ ]:

cluster_to_features = variable_clusters.index.groupby(variable_clusters)

selected_features_to_features = { v[0]:list(v) for v in cluster_to_features.values() }

## keys are the selected feature in the cluster, values are the list of features in the cluster
selected_features_to_features 

Let's package that in a nice little function

In [ ]:
def drop_correlated_features( X , threshold = 0.9 ):
    """
    Args:
        - X (pd.DataFrame) : n,p feature matrix
        - threshold (float) : absolute correlation threshold group variables
    
    Returns:
        - pd.DataFrame : X with only the selected variables
        - dict : keys are the selected features , values are the list of features in the corresponding feature cluster 
    """
    
    corr_threshold = 0.9

    metric = 1 - X.corr().abs()

    HC = AgglomerativeClustering( n_clusters=None , metric='precomputed', linkage = 'single' , distance_threshold = (1-corr_threshold) )
    HC.fit(metric)

    variable_clusters = pd.Series( HC.labels_  , index = X.columns)

    cluster_to_features = variable_clusters.index.groupby(variable_clusters)

    ## keys are the selected feature in the cluster, values are the list of features in the cluster
    selected_features_to_features = { v[0]:list(v) for v in cluster_to_features.values() }

    return X.loc[:,selected_features_to_features.keys()] ,  selected_features_to_features 

In [ ]:
Xs , selected_features_to_features  = drop_correlated_features( X , threshold = 0.9 )
Xs.shape

In [ ]:
import matplotlib.pyplot as plt
C = np.array( Xs.corr() )
sns.histplot( C[ *( np.tril_indices_from( C ,  k = -1 ) ) ] )
plt.xlabel("feature correlation")

In [ ]:
sns.clustermap(Xs.corr(), square=True, cmap="RdBu_r", vmin=-1, vmax=1,
            xticklabels=False, yticklabels=False)